# Figure 10 Variant: Exact Actual-Step Adam Controller

This notebook reruns the architecture-grid spike experiment with one change: the `\rho_a` controller is computed along Adam's actual upcoming step direction, not along the raw gradient direction. It uses the same released `archgrid` setup, but restricts to `Transformer` and `MLP+LN` and uses a `10000x` spike at step 50.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys


def in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except ImportError:
        return False


def find_repo_root():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "src" / "ghosts").exists() and (
            (base / "tutorials").exists() or (base / "experiments").exists()
        ):
            return base

    if in_colab():
        repo = Path('/content/ghosts-of-softmax')
        if not repo.exists():
            subprocess.run(
                [
                    'git', 'clone', '--depth', '1',
                    'https://github.com/piyush314/ghosts-of-softmax.git',
                    str(repo),
                ],
                check=True,
            )
        return repo

    raise RuntimeError(
        'Run this notebook from inside the ghosts-of-softmax repository, '
        'or open it in Google Colab so the setup cell can clone the repo automatically.'
    )


REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


def load_module(name, relative_path):
    path = REPO / relative_path
    module_dir = str(path.parent)
    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


OUTPUT_ROOT = Path('/tmp/ghosts-of-softmax-notebooks')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {REPO}")
print(f"Notebook outputs: {OUTPUT_ROOT}")


In [ ]:
from IPython.display import Image, display
import math
import numpy as np
import torch

arch = load_module('fig10_exact_arch_run', 'experiments/archgrid/run.py')


def _step_to_int(step_obj):
    if torch.is_tensor(step_obj):
        return int(step_obj.item())
    return int(step_obj)


def compute_adam_applied_step(model, opt, lr_override=None):
    group_by_param = {}
    for group in opt.param_groups:
        for p in group['params']:
            group_by_param[p] = group

    vecs = []
    for p in model.parameters():
        if p not in group_by_param or p.grad is None:
            vecs.append(torch.zeros(p.numel(), device=p.device, dtype=p.dtype))
            continue

        group = group_by_param[p]
        beta1, beta2 = group['betas']
        eps = float(group['eps'])
        lr = float(lr_override if lr_override is not None else group['lr'])
        amsgrad = bool(group.get('amsgrad', False))
        maximize = bool(group.get('maximize', False))

        g = p.grad.detach()
        if maximize:
            g = -g

        state = opt.state.get(p, {})
        if 'exp_avg' in state:
            m_prev = state['exp_avg']
            v_prev = state['exp_avg_sq']
            step_prev = _step_to_int(state.get('step', 0))
            v_max_prev = state.get('max_exp_avg_sq', None)
        else:
            m_prev = torch.zeros_like(p)
            v_prev = torch.zeros_like(p)
            step_prev = 0
            v_max_prev = None

        step_t = step_prev + 1
        m_t = beta1 * m_prev + (1.0 - beta1) * g
        v_t = beta2 * v_prev + (1.0 - beta2) * (g * g)
        if amsgrad:
            v_den = v_t if v_max_prev is None else torch.maximum(v_max_prev, v_t)
        else:
            v_den = v_t

        bc1 = 1.0 - (beta1 ** step_t)
        bc2 = 1.0 - (beta2 ** step_t)
        denom = v_den.sqrt() / math.sqrt(max(bc2, 1e-16)) + eps
        adam_term = (m_t / max(bc1, 1e-16)) / denom
        applied = lr * adam_term
        vecs.append(applied.flatten())

    return torch.cat(vecs)


def train_step_rho_exact(model, opt, X, y, lr):
    opt.zero_grad(set_to_none=True)
    logits = model(X)
    loss = torch.nn.functional.cross_entropy(logits, y)
    loss.backward()

    step_vec_unit = compute_adam_applied_step(model, opt, lr_override=1.0)
    unit_norm = float(step_vec_unit.norm().item())
    if unit_norm < 1e-12:
        for pg in opt.param_groups:
            pg['lr'] = lr
        opt.step()
        with torch.no_grad():
            acc = (logits.argmax(1) == y).float().mean().item()
        return {'loss': loss.item(), 'acc': acc, 'rhoA': float('inf'), 'tau': 0.0}

    v = -step_vec_unit / step_vec_unit.norm()
    rhoA = arch.compute_rhoA_jvp(model, X, y, v)
    lr_eff = min(lr, rhoA / unit_norm)
    for pg in opt.param_groups:
        pg['lr'] = lr_eff
    tau = lr_eff * unit_norm
    opt.step()

    with torch.no_grad():
        acc = (logits.argmax(1) == y).float().mean().item()
    return {'loss': loss.item(), 'acc': acc, 'rhoA': rhoA, 'tau': tau}


def run_single_exact(arch_name, seed, steps, base_lr, spike_at, spike_mul, device):
    np.random.seed(seed)
    torch.manual_seed(seed)

    X_np, y_np = arch.load_digits()
    X = torch.tensor(X_np, dtype=torch.float32, device=device)
    y = torch.tensor(y_np, dtype=torch.long, device=device)

    n_test = int(0.2 * len(X))
    Xtr, Xte = X[n_test:], X[:n_test]
    ytr, yte = y[n_test:], y[:n_test]

    results = {
        'plain': {'loss': [], 'acc': []},
        'grad_clip': {'loss': [], 'acc': []},
        'rho_ctrl': {'loss': [], 'acc': [], 'rhoA': [], 'tau': []},
    }

    torch.manual_seed(seed)
    model_plain = arch.make_model(arch_name, device)
    opt_plain = torch.optim.Adam(model_plain.parameters(), lr=base_lr)
    for step in range(steps):
        lr = base_lr * spike_mul if step == spike_at else base_lr
        out = arch.train_step_plain(model_plain, opt_plain, Xtr, ytr, lr)
        results['plain']['loss'].append(out['loss'])
        with torch.no_grad():
            acc_te = (model_plain(Xte).argmax(1) == yte).float().mean().item()
        results['plain']['acc'].append(acc_te)

    torch.manual_seed(seed)
    model_clip = arch.make_model(arch_name, device)
    opt_clip = torch.optim.Adam(model_clip.parameters(), lr=base_lr)
    for step in range(steps):
        lr = base_lr * spike_mul if step == spike_at else base_lr
        out = arch.train_step_clip(model_clip, opt_clip, Xtr, ytr, lr, max_norm=1.0)
        results['grad_clip']['loss'].append(out['loss'])
        with torch.no_grad():
            acc_te = (model_clip(Xte).argmax(1) == yte).float().mean().item()
        results['grad_clip']['acc'].append(acc_te)

    torch.manual_seed(seed)
    model_rho = arch.make_model(arch_name, device)
    opt_rho = torch.optim.Adam(model_rho.parameters(), lr=base_lr)
    for step in range(steps):
        lr = base_lr * spike_mul if step == spike_at else base_lr
        out = train_step_rho_exact(model_rho, opt_rho, Xtr, ytr, lr)
        results['rho_ctrl']['loss'].append(out['loss'])
        results['rho_ctrl']['rhoA'].append(out['rhoA'])
        results['rho_ctrl']['tau'].append(out['tau'])
        with torch.no_grad():
            acc_te = (model_rho(Xte).argmax(1) == yte).float().mean().item()
        results['rho_ctrl']['acc'].append(acc_te)

    return results


def run_experiment_exact(archs, seeds, steps, base_lr, spike_at, spike_mul, device):
    data = {arch_name: {} for arch_name in archs}
    for arch_name in archs:
        print(f'  {arch_name}')
        for i, seed in enumerate(seeds):
            print(f'    seed {seed} ({i + 1}/{len(seeds)})')
            data[arch_name][seed] = run_single_exact(arch_name, seed, steps, base_lr, spike_at, spike_mul, device)
    return data


device = torch.device('cpu')
archs = ['Transformer', 'MLP+LN']
seeds = [0, 1, 2, 3, 4]
steps = 200
base_lr = 0.015
spike_at = 50
spike_mul = 10000.0

data = run_experiment_exact(archs, seeds, steps, base_lr, spike_at, spike_mul, device)

out_dir = OUTPUT_ROOT / 'fig10_exact_actual_adam'
out_dir.mkdir(parents=True, exist_ok=True)
out_png = out_dir / 'archgrid_exact_actual_10000x.png'
out_pdf = out_dir / 'archgrid_exact_actual_10000x.pdf'
out_pt = out_dir / 'archgrid_multiseed_exact_actual_10000x.pt'
out_summary = out_dir / 'summary_exact_actual_10000x.json'

arch.make_plot(data, archs, steps, spike_at, out_png, out_pdf, spike_mul)
summary = arch.build_summary(
    data,
    archs,
    {
        'seeds': seeds,
        'archs': archs,
        'steps': steps,
        'base_lr': base_lr,
        'spike_at': spike_at,
        'spike_mul': spike_mul,
        'controller': 'exact_actual_adam_step',
    },
    out_pt,
    out_png,
    out_pdf,
    out_summary,
)
torch.save({'config': summary['config'], 'data': data}, out_pt)
arch.write_summary(out_summary, summary)

for arch_name in archs:
    finals = [data[arch_name][seed]['rho_ctrl']['acc'][-1] for seed in seeds]
    print(f"{arch_name}: exact controller final acc mean = {np.mean(finals):.3f}")

display(Image(filename=str(out_png)))
